**NOTE** `phimats` environment should be used as kernel

In [1]:
import numpy as np

from PreProcessing import PhysicsConfig, MeshConfig, PreProcessing as PP

from MeshManager import MeshManager

from BoundaryConditions import *

from PostProcessing import WriteXDMF

%load_ext autoreload
%autoreload 2

### Simulation data

In [17]:
SimulName = "SRB_Precharge"
# Element sets
nElementSets = 1
# Number of steps to achieve the load
nSteps = 100

In [18]:
mesh = meshio.read("../SmoothAxi.msh")
nodCoord = mesh.points[:,0:2]          # node coordinates
elemNodeConn = mesh.cells_dict['quad']        # Nodes of all elements

### Read mesh file

In [ ]:
# Element name
elementName = "quad"  		# meshio compatible element name
mesh = MeshManager("../SmoothAxi.msh", elementName)
mesh.WriteMesh("../SmoothAxi")

In [20]:
# Create the config object
meshConfig = MeshConfig(
    nTotNodes=mesh.get_nTotNodes(),
    nTotElements=mesh.get_nTotElements(),
    nDim=mesh.get_nDim(),
    materialNames=mesh.getMaterialNames(),
)

### Apply diffusion boundary conditions

In [21]:
PH2 = 30e6  # Hydrogen gas pressure Pa
S_sieverts = 6.2e-6 # Sieverts' constant mol/(m³.Pa^0.5)

cL = S_sieverts*np.sqrt(PH2) # Surface concnetration in mol/m³

In [ ]:
# Dirichlet BCs list
conBCs = []
exitNods = []

# Surface con
conBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("Surface"), dof=0, value=cL))

# Write external 
WriteBCVTK(SimulName, mesh, conBCs, dofNames=['con'])

In [23]:
conBCs = [[row[0], row[-1]] for row in conBCs]

### Diffusion material data

In [ ]:
# Trapping --------------

Erho = 20000  # Hydroge dislocation enthalpy (positive) [J/mol]

Vm = 7.09e-6  # Molar volume of Fe [m³/mol]
Vrho = 2*Vm   # Molar volume around dislocation  [m³/mol]
Vh = 0        # Partial molar volume of hydrogen in Fe [m³]
			  # Set to zero 
N = 6          
R = 8.31      # Universal gas constant [J/(molK)]
T = 300       # Temperature [K]

theta_b = cL*Vm/N
print(theta_b)

Krho   = np.exp(Erho/(R*T))
theta_rho = theta_b*Krho 

Vrho = 2*Vm
c_rho = theta_rho/Vrho
print("c_rho", c_rho)
Z_rho = c_rho/cL
print("Z_rho", Z_rho)
# zeta_rho = R*T*np.log(Z_rho)
zeta_rho = 0 # Set to zero 
print("zeta_rho", zeta_rho)

In [ ]:
# Diffusivity --------------

dt = 18    # Time increment [s]
s = 1      # Trapping capacity 
m = 0      # Dislocation diffusivity ratio

D0x = 8.45e-8; DQx = 15000  # Diffusivity m^2/s
DL = D0x*np.exp(-DQx/(R*T))

Zd = 0

print("Diffusivity: ", DL)

In [ ]:
DiffMaterial = {
    "SmoothAxi" : {
		"D0x" : D0x,
		"D0y" : D0x,
		"DQx" : DQx,
		"DQy" : DQx,
		"m"   : m,
		"s"   : s,
		"Vh"  : Vh,
		"zeta_rho"   : zeta_rho,
		"Zd"  : Zd,
    }
}

DiffMaterial

### Initialize simulation object

### Diffusion

In [27]:
# Create the config object
diffPhysConfig = PhysicsConfig(
    SimulName = SimulName,
    PhysicsType = "Transport",
    PhysicsCategory = "MechTrapping",
    nSteps    = nSteps,
    dt = dt,
    conB=cL,
    presBCs=conBCs,
)

In [ ]:
DiffNotch = PP(diffPhysConfig, meshConfig, DiffMaterial)

In [ ]:
DiffNotch.WriteInputFile()

In [ ]:
DiffNotch.WriteOutputFile(overwrite=True, AVCON=True)

### Write xdmf file

In [ ]:
WriteXDMF(SimulName,       # Base simulation name
          "../SmoothAxi",  # Mesh file name
          "quadAxi",       # Element name
          nSteps+1,        # Number of steps 
          components=["diff"], # Physics
    	  nDim=2)